# FS10 — SIE vs GMT Slowdown Comparison

Compare slowdowns in September sea ice extent (SIE) versus yearly mean
global mean temperature (GMT) across the CESM2-LE ensemble.

**Prerequisites**
- `scripts/02_cesm2le_slowdowns.py` — SIE slowdown classification
- `scripts/02_cesm2le_slowdowns_gmt.py` — GMT slowdown classification

In [ ]:
import sys
from pathlib import Path
import numpy as np
import xarray as xr
import matplotlib as mpl
import matplotlib.pyplot as plt

PROJECT_ROOT = Path.cwd().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from configs import paths

In [ ]:
# ── Configuration ──────────────────────────────────────────────────────────────
SIE_MONTH     = 'SEP'
START_YEAR    = 1990
END_YEAR      = 2100

mpl.rcParams.update({
    'font.size'        : 18,
    'axes.titlesize'   : 20,
    'axes.labelsize'   : 18,
    'xtick.labelsize'  : 16,
    'ytick.labelsize'  : 16,
    'legend.fontsize'  : 16,
    'figure.titlesize' : 22,
})

## Load SIE and GMT slowdown data

In [ ]:
# ── Load SIE slowdowns ─────────────────────────────────────────────────────────
sie_path = paths.cesm2le_slowdown_file('sie', SIE_MONTH, START_YEAR, END_YEAR)
print(f'SIE slowdown file: {sie_path}')
if not sie_path.exists():
    raise FileNotFoundError(
        f'SIE slowdown file not found: {sie_path}\n'
        f'Run scripts/02_cesm2le_slowdowns.py first.'
    )

ds_sie = xr.open_dataset(sie_path)
sie_slowdown = ds_sie['slowdown'].values      # (nens, n_trends)
sie_years    = ds_sie['nyr'].values            # trend window start years
ds_sie.close()

print(f'  SIE slowdown shape: {sie_slowdown.shape}')
print(f'  SIE year range    : {sie_years[0]}-{sie_years[-1]}')
print(f'  SIE slowdown freq : {sie_slowdown.sum()} / {sie_slowdown.size} '
      f'({100 * sie_slowdown.mean():.1f}%)')

In [ ]:
# ── Load GMT slowdowns ─────────────────────────────────────────────────────────
gmt_path = paths.cesm2le_gmt_slowdown_file(START_YEAR, END_YEAR)
print(f'GMT slowdown file: {gmt_path}')
if not gmt_path.exists():
    raise FileNotFoundError(
        f'GMT slowdown file not found: {gmt_path}\n'
        f'Run scripts/02_cesm2le_slowdowns_gmt.py first.'
    )

ds_gmt = xr.open_dataset(gmt_path)
gmt_slowdown = ds_gmt['slowdown'].values      # (nens, n_trends)
gmt_years    = ds_gmt['nyr'].values            # trend window start years
ds_gmt.close()

print(f'  GMT slowdown shape: {gmt_slowdown.shape}')
print(f'  GMT year range    : {gmt_years[0]}-{gmt_years[-1]}')
print(f'  GMT slowdown freq : {gmt_slowdown.sum()} / {gmt_slowdown.size} '
      f'({100 * gmt_slowdown.mean():.1f}%)')

## Align dimensions and compute BOTH

In [ ]:
# ── Align SIE and GMT on the same trend-window start years ─────────────────────
# Both should use the same window length and start year, but let's be safe.
common_years = np.intersect1d(sie_years, gmt_years)
print(f'Common year range: {common_years[0]}-{common_years[-1]} '
      f'({len(common_years)} windows)')

# Index into each array
sie_idx = np.isin(sie_years, common_years)
gmt_idx = np.isin(gmt_years, common_years)

sie_aligned = sie_slowdown[:, sie_idx]   # (nens, n_common)
gmt_aligned = gmt_slowdown[:, gmt_idx]   # (nens, n_common)

# Sanity check: ensemble and time dims must match
assert sie_aligned.shape == gmt_aligned.shape, (
    f'Shape mismatch after alignment: SIE {sie_aligned.shape} vs GMT {gmt_aligned.shape}'
)
nens, n_common = sie_aligned.shape
print(f'Aligned shape: ({nens}, {n_common})')

# ── BOTH: SIE slowdown AND GMT slowdown in same member/year ───────────────────
both_aligned = (sie_aligned == 1) & (gmt_aligned == 1)

# ── Count number of ensemble members with a slowdown per year ─────────────────
sie_count  = sie_aligned.sum(axis=0)    # (n_common,)
gmt_count  = gmt_aligned.sum(axis=0)    # (n_common,)
both_count = both_aligned.sum(axis=0)   # (n_common,)

print(f'\nPer-year member counts (mean +/- std):')
print(f'  SIE  : {sie_count.mean():.1f} +/- {sie_count.std():.1f}')
print(f'  GMT  : {gmt_count.mean():.1f} +/- {gmt_count.std():.1f}')
print(f'  BOTH : {both_count.mean():.1f} +/- {both_count.std():.1f}')

## Figure: SIE vs GMT slowdown frequency

In [ ]:
fig, ax = plt.subplots(figsize=(14, 6))

ax.plot(common_years, sie_count,  color='steelblue', linewidth=2,
        label='September SIE slowdowns')
ax.plot(common_years, gmt_count,  color='firebrick', linewidth=2,
        label='Yearly GMT slowdowns')
ax.plot(common_years, both_count, color='mediumpurple', linewidth=2,
        label='Both (SIE & GMT)')

ax.set_xlabel('Trend window start year')
ax.set_ylabel('Number of ensemble members')
ax.set_title('Slowdown frequency across CESM2-LE ensemble')
ax.legend(loc='best', framealpha=0.9)
ax.set_xlim(common_years[0], common_years[-1])
ax.set_ylim(bottom=0)

ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

plt.tight_layout()
plt.show()

## Summary statistics

In [ ]:
# Correlation between SIE and GMT slowdown counts
from scipy import stats

r, p = stats.pearsonr(sie_count, gmt_count)
print(f'Pearson r (SIE vs GMT counts): {r:.3f}  (p = {p:.2e})')

# What fraction of SIE slowdowns also have GMT slowdowns?
n_sie_total  = int(sie_aligned.sum())
n_both_total = int(both_aligned.sum())
print(f'\nSIE slowdowns that are also GMT slowdowns: '
      f'{n_both_total} / {n_sie_total} = {100 * n_both_total / n_sie_total:.1f}%')

# What fraction of GMT slowdowns also have SIE slowdowns?
n_gmt_total = int(gmt_aligned.sum())
print(f'GMT slowdowns that are also SIE slowdowns: '
      f'{n_both_total} / {n_gmt_total} = {100 * n_both_total / n_gmt_total:.1f}%')